In [ ]:
#CELL0
from google.colab import drive
drive.mount('/content/drive')

import subprocess
import sys

# Install PyTorch Geometric and dependencies
def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                          package, "--quiet"])

import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION  = torch.version.cuda.replace('.','')

print(f"PyTorch : {TORCH_VERSION}")
print(f"CUDA    : {CUDA_VERSION}")

# Install PyG
subprocess.run([
    "pip", "install", "-q",
    "torch-scatter", "torch-sparse", "torch-cluster",
    "torch-spline-conv", "torch-geometric",
    "-f", f"https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html"
], check=False)

import torch_geometric
print(f"PyG     : {torch_geometric.__version__}")
print("\n✓ Installation OK")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch : 2.11.0
CUDA    : 128
PyG     : 2.8.0

✓ Installation OK


In [ ]:
#CELL1
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, GATv2Conv
from torch_geometric.utils import add_self_loops

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠ No GPU — Runtime → Change runtime type → T4 GPU")

print("\n✓ Imports OK")

Device : cuda
GPU    : NVIDIA L4
VRAM   : 23.7 GB

✓ Imports OK


In [ ]:
#CELL2
DRIVE_DIR  = Path("/content/drive/MyDrive")

DATA_DIR   = DRIVE_DIR / "data_4x"
OUT_DIR    = DRIVE_DIR / "aneo_output"
MODEL_DIR  = DRIVE_DIR / "aneo_models"
TENSOR_DIR = DRIVE_DIR / "aneo_tensors"
GRAPH_DIR  = DRIVE_DIR / "aneo_graphs"   # new: stores graph data
FIG_DIR    = DRIVE_DIR / "aneo_figures"

for d in [OUT_DIR, MODEL_DIR, TENSOR_DIR, GRAPH_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"split.json exists       : {(TENSOR_DIR/'split.json').exists()}")
print(f"normalization stats     : {(OUT_DIR/'normalization_stats.csv').exists()}")
print(f"grid_features.npy       : {(TENSOR_DIR/'grid_features.npy').exists()}")
tensor_files = sorted(TENSOR_DIR.glob("*.npy"))
print(f"Tensor files (.npy)     : {len(tensor_files)}")
print(f"\n✓ Paths OK")

split.json exists       : True
normalization stats     : True
grid_features.npy       : True
Tensor files (.npy)     : 241

✓ Paths OK


In [ ]:
#cell 3
with open(TENSOR_DIR / "split.json", "r") as f:
    split = json.load(f)

TRAIN_DAYS  = split["train_days"]
VAL_DAYS    = split["val_days"]
TEST_DAYS   = split["test_days"]
HEURES      = split["heures"]
NX          = split["nx"]        # 1000
NY          = split["ny"]        # 500
N_INPUT     = split["n_input"]   # 8
N_TARGET    = split["n_target"]  # 3

norm_stats = pd.read_csv(OUT_DIR / "normalization_stats.csv", index_col=0)

STD_U = norm_stats.loc["wind_u_m_s", "std"]
STD_V = norm_stats.loc["wind_v_m_s", "std"]
STD_W = norm_stats.loc["wind_w_m_s", "std"]
MU_U  = norm_stats.loc["wind_u_m_s", "mean"]
MU_V  = norm_stats.loc["wind_v_m_s", "mean"]
MU_W  = norm_stats.loc["wind_w_m_s", "mean"]

# ── GNN config ────────────────────────────────────────────────────────────────
PATCH_SIZE  = 100   # 100×100 nodes per patch = 10,000 nodes
PATCH_STEP  = 100   # no overlap — faster training
K_NEIGHBORS = 8     # each node connects to 8 nearest neighbors
N_NODE_FEAT = 18    # 8 original + 8 grid + 2 temporal
N_EDGE_FEAT = 7     # dx, dy, dz, dist, delta_z0, delta_slope, is_same_terrain
BATCH_SIZE  = 2     # 2 patches × 10,000 nodes = 20,000 nodes per batch

# Derived: number of patches per snapshot
N_PATCHES_X = NX // PATCH_STEP   # 1000 // 100 = 10
N_PATCHES_Y = NY // PATCH_STEP   # 500  // 100 = 5
N_PATCHES   = N_PATCHES_X * N_PATCHES_Y  # 50 patches per snapshot

print("Configuration :\n")
print(f"  Train : {len(TRAIN_DAYS)} days")
print(f"  Val   : {len(VAL_DAYS)} days")
print(f"  Test  : {len(TEST_DAYS)} days")
print(f"  Grid  : {NX} x {NY} = {NX*NY:,} nodes total")
print(f"\n  PATCH_SIZE  : {PATCH_SIZE}×{PATCH_SIZE} = {PATCH_SIZE**2:,} nodes/patch")
print(f"  PATCH_STEP  : {PATCH_STEP}  (no overlap)")
print(f"  N_PATCHES   : {N_PATCHES_X}×{N_PATCHES_Y} = {N_PATCHES} patches/snapshot")
print(f"  BATCH_SIZE  : {BATCH_SIZE} patches → {BATCH_SIZE*PATCH_SIZE**2:,} nodes/batch")
print(f"  K_NEIGHBORS : {K_NEIGHBORS}")
print(f"  N_NODE_FEAT : {N_NODE_FEAT}")
print(f"  N_EDGE_FEAT : {N_EDGE_FEAT}")
print(f"\n  Train items : 96 × {N_PATCHES} = {96*N_PATCHES:,}")
print(f"  Train batches/epoch : {96*N_PATCHES//BATCH_SIZE:,}")
print(f"\n  STD u,v,w : {STD_U:.4f}, {STD_V:.4f}, {STD_W:.4f}")
print(f"  MU  u,v,w : {MU_U:.4f}, {MU_V:.4f}, {MU_W:.4f}")
print(f"\n✓ Config OK")

Configuration :

  Train : 19 days
  Val   : 5 days
  Test  : 6 days
  Grid  : 1000 x 500 = 500,000 nodes total

  PATCH_SIZE  : 100×100 = 10,000 nodes/patch
  PATCH_STEP  : 100  (no overlap)
  N_PATCHES   : 10×5 = 50 patches/snapshot
  BATCH_SIZE  : 2 patches → 20,000 nodes/batch
  K_NEIGHBORS : 8
  N_NODE_FEAT : 18
  N_EDGE_FEAT : 7

  Train items : 96 × 50 = 4,800
  Train batches/epoch : 2,400

  STD u,v,w : 2.0169, 2.1665, 0.0621
  MU  u,v,w : 1.5670, 4.1370, -0.0272

✓ Config OK


In [ ]:
#CELL4
# ── Build patch graph structure from grid points ───────────────────────────────
# This cell runs ONCE and saves the graph topology to Drive.
# The topology (edges, positions) is fixed — only node features change per snapshot.

from sklearn.neighbors import NearestNeighbors

print("Loading grid points...")
grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")
grid = grid.sort_values(['iy', 'ix']).reset_index(drop=True)

print(f"Grid shape : {grid.shape}")
print(f"Columns    : {grid.columns.tolist()}")

# Node positions (normalized)
pos_x = (grid['x_m'].values / grid['x_m'].max()).astype(np.float32)
pos_y = (grid['y_m'].values / grid['y_m'].max()).astype(np.float32)
pos_z = ((grid['z_terrain_m'].values - grid['z_terrain_m'].mean()) /
         (grid['z_terrain_m'].std() + 1e-8)).astype(np.float32)

positions = np.stack([pos_x, pos_y, pos_z], axis=1)  # (N, 3)

# Load static grid features
gf = np.load(TENSOR_DIR / "grid_features.npy")  # (NY, NX, 8)
grid_features_flat = gf.reshape(-1, 8).astype(np.float32)  # (N, 8)

print(f"\nBuilding K={K_NEIGHBORS} nearest neighbor graph...")
print(f"Total nodes : {len(positions):,}")

# Build KNN graph using spatial positions
nbrs = NearestNeighbors(
    n_neighbors=K_NEIGHBORS + 1,  # +1 because self is included
    algorithm='ball_tree',
    metric='euclidean'
).fit(positions[:, :2])  # use x,y only for neighbors (not z)

distances, indices = nbrs.kneighbors(positions[:, :2])

# Build edge index and edge features
print("Building edge features...")
edge_src = []
edge_dst = []
edge_feats = []

for i in range(len(positions)):
    for j_idx in range(1, K_NEIGHBORS + 1):  # skip self (index 0)
        j = indices[i, j_idx]
        edge_src.append(i)
        edge_dst.append(j)

        # Edge features
        dx = positions[j, 0] - positions[i, 0]  # normalized x displacement
        dy = positions[j, 1] - positions[i, 1]  # normalized y displacement
        dz = positions[j, 2] - positions[i, 2]  # normalized z displacement
        dist = distances[i, j_idx]               # euclidean distance
        delta_z0    = grid_features_flat[j, 2] - grid_features_flat[i, 2]  # roughness diff
        delta_slope = grid_features_flat[j, 3] - grid_features_flat[i, 3]  # slope diff
        same_terrain = float(
            grid['terrain_type'].iloc[i] == grid['terrain_type'].iloc[j]
        ) if 'terrain_type' in grid.columns else 0.0

        edge_feats.append([dx, dy, dz, dist, delta_z0, delta_slope, same_terrain])

edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
edge_attr  = torch.tensor(edge_feats, dtype=torch.float32)
node_pos   = torch.tensor(positions, dtype=torch.float32)
node_grid  = torch.tensor(grid_features_flat, dtype=torch.float32)

print(f"\nEdge index shape : {edge_index.shape}")
print(f"Edge attr  shape : {edge_attr.shape}")
print(f"Node pos   shape : {node_pos.shape}")
print(f"Node grid  shape : {node_grid.shape}")

# Save graph topology
torch.save({
    'edge_index'  : edge_index,
    'edge_attr'   : edge_attr,
    'node_pos'    : node_pos,
    'node_grid'   : node_grid,
    'K'           : K_NEIGHBORS,
    'N_nodes'     : len(positions),
    'NX'          : NX,
    'NY'          : NY,
}, GRAPH_DIR / "graph_topology.pt")

print(f"\n✓ Graph topology saved → {GRAPH_DIR/'graph_topology.pt'}")
print(f"  File size: {(GRAPH_DIR/'graph_topology.pt').stat().st_size/1e6:.1f} MB")

Loading grid points...
Grid shape : (500000, 15)
Columns    : ['point_id', 'dataset_name', 'grid_name', 'ix', 'iy', 'x_m', 'y_m', 'z_terrain_m', 'z_rel_m', 'z0_m', 'terrain_type', 'land_use_code', 'slope_deg', 'aspect_deg', 'geom_wkt']

Building K=8 nearest neighbor graph...
Total nodes : 500,000
Building edge features...

Edge index shape : torch.Size([2, 4000000])
Edge attr  shape : torch.Size([4000000, 7])
Node pos   shape : torch.Size([500000, 3])
Node grid  shape : torch.Size([500000, 8])

✓ Graph topology saved → /content/drive/MyDrive/aneo_graphs/graph_topology.pt
  File size: 198.0 MB


In [ ]:
#CELL 5
# ── Load pre-built graph topology ─────────────────────────────────────────────
topo = torch.load(GRAPH_DIR / "graph_topology.pt", weights_only=False)

EDGE_INDEX = topo['edge_index']   # (2, N*K)
EDGE_ATTR  = topo['edge_attr']    # (N*K, 7)
NODE_POS   = topo['node_pos']     # (N, 3)
NODE_GRID  = topo['node_grid']    # (N, 8)
N_NODES    = topo['N_nodes']      # 500,000

print(f"Graph topology loaded :")
print(f"  Nodes      : {N_NODES:,}")
print(f"  Edges      : {EDGE_INDEX.shape[1]:,}")
print(f"  Edge feats : {EDGE_ATTR.shape[1]}")
print(f"  Grid feats : {NODE_GRID.shape[1]}")
print(f"\n✓ Topology ready")

Graph topology loaded :
  Nodes      : 500,000
  Edges      : 4,000,000
  Edge feats : 7
  Grid feats : 8

✓ Topology ready


In [ ]:
#cell 6
class ANEOGraphDataset(Dataset):
    """
    Patch-based GNN dataset — 100×100 node patches.
    Uses fully vectorized edge pre-computation (no Python loops over edges).
    """

    def __init__(self, days, heures, tensor_dir, patch_size=100,
                 patch_step=100, augment=False):
        self.tensor_dir  = Path(tensor_dir)
        self.patch_size  = patch_size
        self.patch_step  = patch_step
        self.augment     = augment
        self.snapshots   = []

        for day in days:
            for h in heures:
                x_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_X.npy"
                y_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_Y.npy"
                if x_path.exists() and y_path.exists():
                    self.snapshots.append((x_path, y_path, day, h))

        # Build patch origins
        self.patch_origins = []
        for iy in range(0, NY - patch_size + 1, patch_step):
            for ix in range(0, NX - patch_size + 1, patch_step):
                self.patch_origins.append((iy, ix))

        self.n_patches = len(self.patch_origins)
        self.length    = len(self.snapshots) * self.n_patches

        # ── Pre-compute patch edges using pure numpy (no Python loops) ────────
        print(f"  Pre-computing edges for {self.n_patches} patches "
              f"(fully vectorized)...")

        src_g = EDGE_INDEX[0].numpy()  # (4M,)
        dst_g = EDGE_INDEX[1].numpy()  # (4M,)

        # Build a boolean membership array: node_id → in_patch
        # For each patch we create a boolean mask over all 500K nodes
        self.patch_edge_data = []

        for p_idx, (iy0, ix0) in enumerate(self.patch_origins):
            iy1 = iy0 + patch_size
            ix1 = ix0 + patch_size

            # Boolean mask over all 500K nodes: True if node is in patch
            # Node global_id = iy * NX + ix
            # A node is in patch if: iy0 <= iy < iy1 AND ix0 <= ix < ix1
            # From global_id: iy = global_id // NX, ix = global_id % NX
            src_iy = src_g // NX;  src_ix = src_g % NX
            dst_iy = dst_g // NX;  dst_ix = dst_g % NX

            src_in = ((src_iy >= iy0) & (src_iy < iy1) &
                      (src_ix >= ix0) & (src_ix < ix1))
            dst_in = ((dst_iy >= iy0) & (dst_iy < iy1) &
                      (dst_ix >= ix0) & (dst_ix < ix1))

            # Edge is internal if both endpoints are in patch
            edge_mask = src_in & dst_in
            valid_idx = np.where(edge_mask)[0]

            # Map global node ids to local (0-based within patch)
            # local_id = (iy - iy0) * patch_size + (ix - ix0)
            src_valid = src_g[valid_idx]
            dst_valid = dst_g[valid_idx]

            src_local = ((src_valid // NX) - iy0) * patch_size + \
                        ((src_valid %  NX) - ix0)
            dst_local = ((dst_valid // NX) - iy0) * patch_size + \
                        ((dst_valid %  NX) - ix0)

            # Global node ids for this patch (for feature lookup)
            iy_range  = np.arange(iy0, iy1)
            ix_range  = np.arange(ix0, ix1)
            iy_g, ix_g = np.meshgrid(iy_range, ix_range, indexing='ij')
            patch_ids = (iy_g * NX + ix_g).flatten()  # (P*P,)

            self.patch_edge_data.append({
                'patch_ids' : patch_ids,
                'src_local' : src_local.astype(np.int64),
                'dst_local' : dst_local.astype(np.int64),
                'valid_idx' : valid_idx,
            })

            if (p_idx + 1) % 10 == 0:
                print(f"    Patch {p_idx+1}/{self.n_patches} done — "
                      f"{len(valid_idx)} edges")

        print(f"\n  Dataset : {len(self.snapshots)} snapshots × "
              f"{self.n_patches} patches = {self.length:,} items "
              f"({'aug ON' if augment else 'aug OFF'})")
        avg_edges = np.mean([len(d['valid_idx'])
                            for d in self.patch_edge_data])
        print(f"  Avg edges/patch : {avg_edges:.0f}")

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        snap_idx  = idx // self.n_patches
        patch_idx = idx %  self.n_patches

        x_path, y_path, day, h = self.snapshots[snap_idx]
        iy0, ix0 = self.patch_origins[patch_idx]
        iy1      = iy0 + self.patch_size
        ix1      = ix0 + self.patch_size
        P        = self.patch_size
        N_patch  = P * P

        # Load snapshot and extract patch
        X_full  = np.load(x_path).astype(np.float32)
        Y_full  = np.load(y_path).astype(np.float32)
        X_flat  = X_full[iy0:iy1, ix0:ix1, :].reshape(N_patch, 8)
        Y_flat  = Y_full[iy0:iy1, ix0:ix1, :].reshape(N_patch, 3)

        # Pre-computed patch data
        pd_       = self.patch_edge_data[patch_idx]
        patch_ids = pd_['patch_ids']
        src_local = pd_['src_local']
        dst_local = pd_['dst_local']
        valid_idx = pd_['valid_idx']

        # Node features: 8 atm + 8 grid + 2 temporal = 18
        grid_feat = NODE_GRID[patch_ids].numpy()
        hour_sin  = float(np.sin(2 * np.pi * h / 24))
        hour_cos  = float(np.cos(2 * np.pi * h / 24))
        temporal  = np.full((N_patch, 2),
                           [hour_sin, hour_cos], dtype=np.float32)
        node_x    = np.concatenate([X_flat, grid_feat, temporal], axis=1)

        # Augmentation
        if self.augment:
            if np.random.rand() > 0.5:
                scale = np.random.uniform(0.8, 1.2)
                node_x[:, 4:7] *= scale
                Y_flat         *= scale
            if np.random.rand() > 0.5:
                node_x[:, :8] += np.random.randn(
                    N_patch, 8).astype(np.float32) * 0.02

        return Data(
            x          = torch.tensor(node_x, dtype=torch.float32),
            edge_index = torch.tensor(
                np.stack([src_local, dst_local]), dtype=torch.long),
            edge_attr  = EDGE_ATTR[valid_idx],
            y          = torch.tensor(Y_flat, dtype=torch.float32),
            day        = torch.tensor([day],  dtype=torch.long),
            hour       = torch.tensor([h],    dtype=torch.long),
            patch_iy0  = torch.tensor([iy0],  dtype=torch.long),
            patch_ix0  = torch.tensor([ix0],  dtype=torch.long),
        )


print("✓ ANEOGraphDataset defined (fully vectorized)")

✓ ANEOGraphDataset defined (fully vectorized)


In [ ]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_x, modes_y):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.modes_x      = modes_x
        self.modes_y      = modes_y
        scale = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels,
                               modes_x, modes_y, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels,
                               modes_x, modes_y, dtype=torch.cfloat))

    def compl_mul2d(self, input, weights):
        return torch.einsum("bixy,ioxy->boxy", input, weights)

    def forward(self, x):
        B = x.shape[0]
        x_ft = torch.fft.rfft2(x.float())
        out_ft = torch.zeros(B, self.out_channels,
                             x.size(-2), x.size(-1)//2+1,
                             dtype=torch.cfloat, device=x.device)
        out_ft[:,:,:self.modes_x,:self.modes_y] = \
            self.compl_mul2d(x_ft[:,:,:self.modes_x,:self.modes_y], self.weights1)
        out_ft[:,:,-self.modes_x:,:self.modes_y] = \
            self.compl_mul2d(x_ft[:,:,-self.modes_x:,:self.modes_y], self.weights2)
        return torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))


class FNOBlock(nn.Module):
    def __init__(self, channels, modes_x, modes_y, dropout=0.05):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes_x, modes_y)
        self.local    = nn.Conv2d(channels, channels, kernel_size=1)
        self.norm     = nn.InstanceNorm2d(channels, affine=True)
        self.dropout  = nn.Dropout2d(p=dropout)

    def forward(self, x):
        return F.gelu(self.dropout(
            self.norm(self.spectral(x) + self.local(x))))


class UFNO2D(nn.Module):
    """
    U-FNO: U-Net + Fourier Neural Operator.
    Encoder-decoder with FNO blocks at each scale + skip connections.
    Separate output heads for u, v, w.

    Architecture:
      Encoder: 3 levels of downsampling + FNO blocks
      Bottleneck: FNO block at coarsest resolution
      Decoder: 3 levels of upsampling + skip connections + FNO blocks
      Heads: separate u/v/w projection heads
    """
    def __init__(self, in_channels, out_channels,
                 base_channels=48, modes_x=20, modes_y=20, dropout=0.05):
        super().__init__()

        c1 = base_channels        # 48  — full resolution
        c2 = base_channels * 2    # 96  — half resolution
        c3 = base_channels * 4    # 192 — quarter resolution

        # ── Input projection ──────────────────────────────────────────────────
        self.input_proj = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=1),
            nn.GELU(),
        )

        # ── Encoder ───────────────────────────────────────────────────────────
        self.enc1 = FNOBlock(c1, modes_x, modes_y, dropout)
        self.down1 = nn.Conv2d(c1, c2, kernel_size=2, stride=2)  # /2

        self.enc2 = FNOBlock(c2, modes_x//2, modes_y//2, dropout)
        self.down2 = nn.Conv2d(c2, c3, kernel_size=2, stride=2)  # /4

        # ── Bottleneck ────────────────────────────────────────────────────────
        self.bottleneck = nn.Sequential(
            FNOBlock(c3, modes_x//4, modes_y//4, dropout),
            FNOBlock(c3, modes_x//4, modes_y//4, dropout),
        )

        # ── Decoder ───────────────────────────────────────────────────────────
        self.up2   = nn.ConvTranspose2d(c3, c2, kernel_size=2, stride=2)
        self.dec2  = FNOBlock(c2*2, modes_x//2, modes_y//2, dropout)
        self.proj2 = nn.Conv2d(c2*2, c2, kernel_size=1)

        self.up1   = nn.ConvTranspose2d(c2, c1, kernel_size=2, stride=2)
        self.dec1  = FNOBlock(c1*2, modes_x, modes_y, dropout)
        self.proj1 = nn.Conv2d(c1*2, c1, kernel_size=1)

        # ── Separate output heads ─────────────────────────────────────────────
        def make_head(ch):
            return nn.Sequential(
                nn.Conv2d(ch, ch//2, kernel_size=1), nn.GELU(),
                nn.Conv2d(ch//2, ch//4, kernel_size=1), nn.GELU(),
                nn.Conv2d(ch//4, 1, kernel_size=1),
            )

        self.head_u = make_head(c1)
        self.head_v = make_head(c1)
        self.head_w = make_head(c1)

    def forward(self, x):
        # Input projection
        x = self.input_proj(x)          # (B, c1, 500, 1000)

        # Encoder
        e1 = self.enc1(x)               # (B, c1, 500, 1000)
        e2 = self.enc2(self.down1(e1))  # (B, c2, 250, 500)
        e3 = self.down2(e2)             # (B, c3, 125, 250)

        # Bottleneck
        b  = self.bottleneck(e3)        # (B, c3, 125, 250)

        # Decoder with skip connections
        d2 = self.up2(b)                # (B, c2, 250, 500)
        d2 = self.proj2(
            self.dec2(
                torch.cat([d2, e2], dim=1)))  # (B, c2, 250, 500)

        d1 = self.up1(d2)               # (B, c1, 500, 1000)
        d1 = self.proj1(
            self.dec1(
                torch.cat([d1, e1], dim=1)))  # (B, c1, 500, 1000)

        # Separate heads
        return torch.cat([
            self.head_u(d1),
            self.head_v(d1),
            self.head_w(d1),
        ], dim=1)                        # (B, 3, 500, 1000)


model = UFNO2D(
    in_channels   = N_INPUT_EFFECTIVE,  # 18
    out_channels  = N_TARGET,           # 3
    base_channels = 48,                 # c1=48, c2=96, c3=192
    modes_x       = 20,
    modes_y       = 20,
    dropout       = 0.05,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-FNO Model :")
print(f"  in_channels   : {N_INPUT_EFFECTIVE}")
print(f"  base_channels : 48 / 96 / 192  (3 encoder levels)")
print(f"  modes         : 20 / 10 / 5   (coarsening per level)")
print(f"  dropout       : 0.05")
print(f"  skip connects : yes (at each decoder level)")
print(f"  output heads  : separate u / v / w")
print(f"  parameters    : {n_params:,}")
print(f"  device        : {device}")
print(f"  on GPU        : {next(model.parameters()).is_cuda}")
print(f"\n✓ U-FNO ready")

  Pre-computing edges for 50 patches (fully vectorized)...
    Patch 10/50 done — 79491 edges
    Patch 20/50 done — 79033 edges
    Patch 30/50 done — 79226 edges
    Patch 40/50 done — 79389 edges
    Patch 50/50 done — 79355 edges

  Dataset : 96 snapshots × 50 patches = 4,800 items (aug ON)
  Avg edges/patch : 78987
  Pre-computing edges for 50 patches (fully vectorized)...
    Patch 10/50 done — 79491 edges
    Patch 20/50 done — 79033 edges
    Patch 30/50 done — 79226 edges
    Patch 40/50 done — 79389 edges
    Patch 50/50 done — 79355 edges

  Dataset : 20 snapshots × 50 patches = 1,000 items (aug OFF)
  Avg edges/patch : 78987
  Pre-computing edges for 50 patches (fully vectorized)...
    Patch 10/50 done — 79491 edges
    Patch 20/50 done — 79033 edges
    Patch 30/50 done — 79226 edges
    Patch 40/50 done — 79389 edges
    Patch 50/50 done — 79355 edges

  Dataset : 24 snapshots × 50 patches = 1,200 items (aug OFF)
  Avg edges/patch : 78987
Patch size     : 100×100 = 10,00

In [ ]:
#CELL 8
class PhysicsGNNLayer(MessagePassing):
    """
    Single GNN layer using GATv2 attention with edge features.
    Edge features (terrain connectivity, distance, elevation diff)
    are incorporated into the attention mechanism — physically-informed
    message passing.
    """
    def __init__(self, in_channels, out_channels, edge_dim=7,
                 heads=4, dropout=0.05):
        super().__init__(aggr='mean')
        self.heads       = heads
        self.out_channels = out_channels

        # GATv2 attention with edge features
        self.conv = GATv2Conv(
            in_channels  = in_channels,
            out_channels = out_channels // heads,
            heads        = heads,
            edge_dim     = edge_dim,
            dropout      = dropout,
            concat       = True,
        )
        self.norm = nn.LayerNorm(out_channels)
        self.act  = nn.GELU()

        # Residual projection if needed
        self.res_proj = nn.Linear(in_channels, out_channels) \
            if in_channels != out_channels else nn.Identity()

    def forward(self, x, edge_index, edge_attr):
        residual = self.res_proj(x)
        out = self.conv(x, edge_index, edge_attr)
        return self.act(self.norm(out + residual))


class WindGNN(nn.Module):
    """
    Physics-Constrained Graph Neural Network for wind field reconstruction.

    Architecture:
      1. Node feature embedding (18 → hidden_dim)
      2. Edge feature embedding (7 → hidden_dim)
      3. N_LAYERS of PhysicsGNNLayer (message passing with attention)
      4. Separate output heads for u, v, w
    """
    def __init__(self, n_node_feat=18, n_edge_feat=7,
                 hidden_dim=128, n_layers=6,
                 heads=4, dropout=0.05):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.n_layers   = n_layers

        # Input embeddings
        self.node_embed = nn.Sequential(
            nn.Linear(n_node_feat, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
        )
        self.edge_embed = nn.Sequential(
            nn.Linear(n_edge_feat, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
        )

        # GNN layers
        self.layers = nn.ModuleList([
            PhysicsGNNLayer(hidden_dim, hidden_dim, hidden_dim,
                           heads, dropout)
            for _ in range(n_layers)
        ])

        # Separate output heads for each wind component
        def make_head():
            return nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(),
                nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.GELU(),
                nn.Linear(hidden_dim // 4, 1),
            )

        self.head_u = make_head()
        self.head_v = make_head()
        self.head_w = make_head()

    def forward(self, data):
        x          = data.x
        edge_index = data.edge_index
        edge_attr  = data.edge_attr

        # Embed node features
        x = self.node_embed(x)

        # Embed edge features
        edge_attr = self.edge_embed(edge_attr)

        # Message passing layers
        for layer in self.layers:
            x = layer(x, edge_index, edge_attr)

        # Predict each component separately
        u = self.head_u(x)  # (N, 1)
        v = self.head_v(x)  # (N, 1)
        w = self.head_w(x)  # (N, 1)

        return torch.cat([u, v, w], dim=1)  # (N, 3)


model = WindGNN(
    n_node_feat = N_NODE_FEAT,
    n_edge_feat = N_EDGE_FEAT,
    hidden_dim  = 128,
    n_layers    = 6,
    heads       = 4,
    dropout     = 0.05,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"WindGNN Model :")
print(f"  n_node_feat : {N_NODE_FEAT}")
print(f"  n_edge_feat : {N_EDGE_FEAT}")
print(f"  hidden_dim  : 128")
print(f"  n_layers    : 6")
print(f"  heads       : 4  (GATv2 attention)")
print(f"  parameters  : {n_params:,}")
print(f"  device      : {device}")
print(f"\n✓ Model ready")

WindGNN Model :
  n_node_feat : 18
  n_edge_feat : 7
  hidden_dim  : 128
  n_layers    : 6
  heads       : 4  (GATv2 attention)
  parameters  : 334,595
  device      : cuda

✓ Model ready


In [ ]:
#CELL 9
class GNNPhysicsLoss(nn.Module):
    """
    Physics-constrained loss adapted for graph/node-level predictions.
    L = L_data + phys_weight * (lambda_mass * L_mass + lambda_bc * L_bc)

    Node-level mass conservation:
      For each node i, approximate divergence using neighbor differences.
    """
    def __init__(self, lambda_mass=0.3, lambda_bc=0.05, dx=50.0):
        super().__init__()
        self.lambda_mass = lambda_mass
        self.lambda_bc   = lambda_bc
        self.dx          = dx

    def forward(self, pred, target, data, rho, phys_weight=1.0):
        # L_data: MSE over all nodes
        L_data = F.mse_loss(pred, target)

        # L_mass: approximate divergence at each node using neighbor info
        # For graph setting: use edge-based finite differences
        src, dst = data.edge_index
        u_pred = pred[:, 0]
        v_pred = pred[:, 1]
        w_pred = pred[:, 2]

        # du/dx approximated via edge displacement
        dx_edge = data.edge_attr[:, 0]  # normalized x displacement
        dy_edge = data.edge_attr[:, 1]  # normalized y displacement

        du = u_pred[dst] - u_pred[src]
        dv = v_pred[dst] - v_pred[src]

        # Divergence approximation: mean of (du/dx + dv/dy) over edges
        epsilon = 1e-8
        dudx = du / (dx_edge.abs() + epsilon)
        dvdy = dv / (dy_edge.abs() + epsilon)
        divergence = dudx + dvdy
        L_mass = (divergence ** 2).mean()

        # L_bc: w should be near zero at ground-level nodes
        # Identify ground nodes: z_rel_m ≈ 0 (from node features channel 9)
        z_rel = data.x[:, 9]  # z_rel_m normalized
        ground_mask = (z_rel.abs() < 0.1)
        if ground_mask.sum() > 0:
            L_bc = (w_pred[ground_mask] ** 2).mean()
        else:
            L_bc = torch.tensor(0.0, device=pred.device)

        phys_term = self.lambda_mass * L_mass + self.lambda_bc * L_bc
        L_total = L_data + phys_weight * phys_term

        return L_total, L_data, L_mass, L_bc


criterion = GNNPhysicsLoss(lambda_mass=0.3, lambda_bc=0.05, dx=50.0)
print("✓ GNN physics loss ready")

✓ GNN physics loss ready


In [ ]:
#CELL 10
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = 3e-4,
    weight_decay = 1e-4,
    betas        = (0.9, 0.999),
)

N_EPOCHS      = 100
WARMUP_EPOCHS = 10
EARLY_STOP    = 15
ACCUM_STEPS   = 2

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-6
)

def physics_warmup(epoch, warmup_epochs):
    if warmup_epochs <= 0:
        return 1.0
    return min(1.0, (epoch + 1) / warmup_epochs)

print("Training config :")
print(f"  N_EPOCHS      : {N_EPOCHS}")
print(f"  WARMUP_EPOCHS : {WARMUP_EPOCHS}")
print(f"  EARLY_STOP    : {EARLY_STOP}")
print(f"  ACCUM_STEPS   : {ACCUM_STEPS}")
print(f"  Optimizer     : AdamW  lr=3e-4  wd=1e-4")
print(f"\n✓ Ready")

Training config :
  N_EPOCHS      : 100
  WARMUP_EPOCHS : 10
  EARLY_STOP    : 15
  ACCUM_STEPS   : 2
  Optimizer     : AdamW  lr=3e-4  wd=1e-4

✓ Ready


In [ ]:
#CELL 11
def compute_r2_gnn(pred, true):
    """R² per output component (u=0, v=1, w=2) for GNN node predictions."""
    pred_np = pred.detach().cpu().numpy()
    true_np = true.detach().cpu().numpy()
    scores = []
    for c in range(pred_np.shape[1]):
        y_true = true_np[:, c]
        y_pred = pred_np[:, c]
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        scores.append(1 - ss_res / (ss_tot + 1e-8))
    return np.mean(scores), scores


def train_epoch(epoch):
    model.train()
    train_loss = 0.0
    phys_w = physics_warmup(epoch, WARMUP_EPOCHS)
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        batch = batch.to(device)
        rho   = batch.x[:, 1:2]  # density channel

        pred = model(batch)
        loss, l_data, l_mass, l_bc = criterion(
            pred, batch.y, batch, rho, phys_weight=phys_w)
        loss_scaled = loss / ACCUM_STEPS
        loss_scaled.backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item()

    return train_loss / len(train_loader)


def validate_epoch(epoch):
    model.eval()
    val_loss  = 0.0
    r2_means  = []
    r2_ch_sum = np.zeros(3)
    phys_w    = physics_warmup(epoch, WARMUP_EPOCHS)

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            rho   = batch.x[:, 1:2]
            pred  = model(batch)
            loss, _, _, _ = criterion(
                pred, batch.y, batch, rho, phys_weight=phys_w)
            val_loss += loss.item()
            r2m, r2c = compute_r2_gnn(pred, batch.y)
            r2_means.append(r2m)
            r2_ch_sum += np.array(r2c)

    n = len(val_loader)
    return val_loss / n, np.mean(r2_means), r2_ch_sum / n


print("✓ train_epoch / validate_epoch defined")

✓ train_epoch / validate_epoch defined


In [ ]:
#CELL 12
history = {
    'train_loss': [], 'val_loss': [], 'r2': [],
    'r2_u': [], 'r2_v': [], 'r2_w': [], 'lr': []
}

best_val_loss    = float('inf')
best_epoch       = 0
best_r2          = -np.inf
patience_counter = 0

print("="*88)
print("STARTING GNN TRAINING")
print("="*88)
print(f"{'Ep':>4} | {'Train':>9} | {'Val':>9} | {'R²avg':>7} | "
      f"{'R²_u':>6} | {'R²_v':>6} | {'R²_w':>6} | {'φ_w':>5} | {'LR':>8} | Status")
print("-"*88)

try:
    for epoch in range(N_EPOCHS):
        tr_loss             = train_epoch(epoch)
        va_loss, r2, r2_ch  = validate_epoch(epoch)
        scheduler.step()

        lr     = optimizer.param_groups[0]['lr']
        phys_w = physics_warmup(epoch, WARMUP_EPOCHS)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['r2'].append(r2)
        history['r2_u'].append(r2_ch[0])
        history['r2_v'].append(r2_ch[1])
        history['r2_w'].append(r2_ch[2])
        history['lr'].append(lr)

        print(f"{epoch+1:>4} | {tr_loss:>9.5f} | {va_loss:>9.5f} | "
              f"{r2:>7.4f} | {r2_ch[0]:>6.4f} | {r2_ch[1]:>6.4f} | "
              f"{r2_ch[2]:>6.4f} | {phys_w:>5.2f} | {lr:>8.2e} | ", end="")

        if va_loss < best_val_loss:
            best_val_loss    = va_loss
            best_epoch       = epoch + 1
            best_r2          = r2
            patience_counter = 0
            torch.save({
                'epoch'            : epoch + 1,
                'model_state_dict' : model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss'         : va_loss,
                'r2'               : r2,
                'r2_u'             : r2_ch[0],
                'r2_v'             : r2_ch[1],
                'r2_w'             : r2_ch[2],
                'history'          : history,
            }, MODEL_DIR / 'best_gnn_model.pth')
            print("✓ BEST")
        else:
            patience_counter += 1
            print(f"(patience {patience_counter}/{EARLY_STOP})"
                  if patience_counter % 5 == 0 else "")

        if patience_counter >= EARLY_STOP:
            print(f"\n⚠ Early stopping at epoch {epoch+1}")
            break

except KeyboardInterrupt:
    print("\n⚠ Training interrupted")
except Exception as e:
    print(f"\n✗ Training failed: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

print("\n" + "="*88)
print("TRAINING COMPLETE")
print(f"Best epoch : {best_epoch}  "
      f"Best val_loss : {best_val_loss:.6f}  "
      f"Best R² : {best_r2:.4f}")
print("="*88)

pd.DataFrame(history).to_csv(OUT_DIR / 'gnn_training_history.csv', index=False)
print(f"✓ History saved")

STARTING GNN TRAINING
  Ep |     Train |       Val |   R²avg |   R²_u |   R²_v |   R²_w |   φ_w |       LR | Status
----------------------------------------------------------------------------------------
   1 | 448917968.93409 | 550.06750 | -2.2337 | -2.1415 | -3.5043 | -1.0553 |  0.10 | 3.00e-04 | ✓ BEST


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fe5226ec400>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self._selector.poll(timeout)
                    ^^^^^^^^^^^^^


⚠ Training interrupted

TRAINING COMPLETE
Best epoch : 1  Best val_loss : 550.067502  Best R² : -2.2337
✓ History saved


In [ ]:
#CELL 13
# ── Full domain inference: stitch patches back into 500×1000 field ───────────

def predict_full_field(model, day, hour, device):
    """
    Predict u, v, w for the entire 500×1000 domain
    by processing patches and stitching results with overlap averaging.
    """
    model.eval()

    # Load full snapshot
    x_path = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_X.npy"
    y_path = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_Y.npy"
    X_full = np.load(x_path).astype(np.float32)  # (NY, NX, 8)
    Y_full = np.load(y_path).astype(np.float32)  # (NY, NX, 3)

    pred_sum   = np.zeros((NY, NX, 3), dtype=np.float32)
    count_grid = np.zeros((NY, NX),    dtype=np.float32)

    hour_sin = float(np.sin(2 * np.pi * hour / 24))
    hour_cos = float(np.cos(2 * np.pi * hour / 24))

    patch_origins = []
    for iy in range(0, NY - PATCH_SIZE + 1, PATCH_STEP):
        for ix in range(0, NX - PATCH_SIZE + 1, PATCH_STEP):
            patch_origins.append((iy, ix))

    with torch.no_grad():
        for iy0, ix0 in patch_origins:
            iy1, ix1 = iy0 + PATCH_SIZE, ix0 + PATCH_SIZE
            P        = PATCH_SIZE
            N_patch  = P * P

            X_patch  = X_full[iy0:iy1, ix0:ix1, :].reshape(N_patch, 8)

            patch_node_ids = np.array([
                iy * NX + ix
                for iy in range(iy0, iy1)
                for ix in range(ix0, ix1)
            ])

            grid_feat = NODE_GRID[patch_node_ids].numpy()
            temporal  = np.full((N_patch, 2), [hour_sin, hour_cos],
                               dtype=np.float32)
            node_x    = np.concatenate([X_patch, grid_feat, temporal], axis=1)

            # Build local graph
            global_to_local = {gid: lid for lid, gid
                               in enumerate(patch_node_ids)}
            patch_set       = set(patch_node_ids.tolist())
            src_g = EDGE_INDEX[0].numpy()
            dst_g = EDGE_INDEX[1].numpy()
            mask  = np.array([s in patch_set and d in patch_set
                              for s, d in zip(src_g, dst_g)])
            src_l = np.array([global_to_local[s] for s in src_g[mask]])
            dst_l = np.array([global_to_local[d] for d in dst_g[mask]])

            data = Data(
                x          = torch.tensor(node_x, dtype=torch.float32),
                edge_index = torch.tensor(np.stack([src_l, dst_l]),
                                         dtype=torch.long),
                edge_attr  = EDGE_ATTR[mask],
            ).to(device)

            pred = model(data).cpu().numpy()  # (N_patch, 3)
            pred_grid = pred.reshape(P, P, 3)

            pred_sum[iy0:iy1, ix0:ix1, :]  += pred_grid
            count_grid[iy0:iy1, ix0:ix1]   += 1.0

    # Average overlapping predictions
    count_grid = np.maximum(count_grid, 1.0)
    pred_field = pred_sum / count_grid[:, :, np.newaxis]

    return pred_field, Y_full


# ── Run evaluation on test set ─────────────────────────────────────────────────
ckpt = torch.load(MODEL_DIR / "best_gnn_model.pth",
                  map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded epoch {ckpt['epoch']}  "
      f"val_loss={ckpt['val_loss']:.6f}  "
      f"R²avg={ckpt['r2']:.4f}  "
      f"R²_u={ckpt['r2_u']:.4f}  "
      f"R²_v={ckpt['r2_v']:.4f}  "
      f"R²_w={ckpt['r2_w']:.4f}")

results = []
for day in TEST_DAYS:
    for hour in HEURES:
        print(f"  Predicting J{day:3d} h{hour:02d}...", end=" ")
        pred_field, true_field = predict_full_field(model, day, hour, device)

        # Denormalize
        u_p = pred_field[:,:,0]*STD_U+MU_U; v_p = pred_field[:,:,1]*STD_V+MU_V
        w_p = pred_field[:,:,2]*STD_W+MU_W
        u_t = true_field[:,:,0]*STD_U+MU_U; v_t = true_field[:,:,1]*STD_V+MU_V
        w_t = true_field[:,:,2]*STD_W+MU_W

        mae_u  = np.mean(np.abs(u_p-u_t))
        mae_v  = np.mean(np.abs(v_p-v_t))
        mae_w  = np.mean(np.abs(w_p-w_t))
        rmse_u = np.sqrt(np.mean((u_p-u_t)**2))
        rmse_v = np.sqrt(np.mean((v_p-v_t)**2))
        rmse_w = np.sqrt(np.mean((w_p-w_t)**2))
        r2_u   = 1-np.sum((u_p-u_t)**2)/(np.sum((u_t-u_t.mean())**2)+1e-9)
        r2_v   = 1-np.sum((v_p-v_t)**2)/(np.sum((v_t-v_t.mean())**2)+1e-9)
        r2_w   = 1-np.sum((w_p-w_t)**2)/(np.sum((w_t-w_t.mean())**2)+1e-9)

        ws_p    = np.sqrt(u_p**2+v_p**2); ws_t = np.sqrt(u_t**2+v_t**2)
        mae_ws  = np.mean(np.abs(ws_p-ws_t))
        dir_p   = np.degrees(np.arctan2(v_p,u_p))%360
        dir_t   = np.degrees(np.arctan2(v_t,u_t))%360
        dir_err = np.minimum(np.abs(dir_p-dir_t), 360-np.abs(dir_p-dir_t))
        mae_dir = np.mean(dir_err)

        acc_u_05 = (np.abs(u_p-u_t)<0.5).mean()*100
        acc_u_10 = (np.abs(u_p-u_t)<1.0).mean()*100
        acc_v_05 = (np.abs(v_p-v_t)<0.5).mean()*100
        acc_v_10 = (np.abs(v_p-v_t)<1.0).mean()*100
        acc_w_01 = (np.abs(w_p-w_t)<0.1).mean()*100

        results.append(dict(
            day=day, hour=hour,
            mae_u=mae_u, mae_v=mae_v, mae_w=mae_w,
            rmse_u=rmse_u, rmse_v=rmse_v, rmse_w=rmse_w,
            r2_u=r2_u, r2_v=r2_v, r2_w=r2_w,
            mae_ws=mae_ws, mae_dir=mae_dir,
            acc_u_05=acc_u_05, acc_u_10=acc_u_10,
            acc_v_05=acc_v_05, acc_v_10=acc_v_10,
            acc_w_01=acc_w_01,
        ))

        print(f"R² u={r2_u:.4f} v={r2_v:.4f} w={r2_w:.4f} | "
              f"Acc_u={acc_u_05:.1f}% Acc_v={acc_v_05:.1f}%")

df_r = pd.DataFrame(results)
df_r.to_csv(OUT_DIR / 'gnn_evaluation_results.csv', index=False)

print(f"\n{'='*72}")
print(f"GNN TEST SET METRICS (mean over {len(df_r)} snapshots):")
print(f"{'='*72}")
for col in ['mae_u','mae_v','mae_w','rmse_u','rmse_v','rmse_w',
            'r2_u','r2_v','r2_w','mae_ws','mae_dir',
            'acc_u_05','acc_u_10','acc_v_05','acc_v_10','acc_w_01']:
    print(f"  {col:12s}: {df_r[col].mean():.4f}")
print('='*72)
print(f"\n✓ GNN results saved: {OUT_DIR/'gnn_evaluation_results.csv'}")